In [ ]:
import os

SEED = 41
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import random
from GraphST import preprocess
from GraphST.preprocess import compute_moranI_and_filter
import torch
import pandas as pd
import scanpy as sc
import numpy as np
from sklearn import metrics
import matplotlib.pyplot as plt
from GraphST import GraphST
import itertools
from datetime import datetime
import warnings
import csv

warnings.filterwarnings("ignore")


def seed_everything(seed=41):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass

    torch.use_deterministic_algorithms(True, warn_only=False)


seed_everything(SEED)


def select_most_idle_gpu():
    if not torch.cuda.is_available():
        return "cpu"

    num_gpus = torch.cuda.device_count()
    gpu_usage = []

    for i in range(num_gpus):
        try:
            import subprocess

            cmd = (
                "nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total "
                f"--format=csv,noheader,nounits -i {i}"
            )
            result = subprocess.check_output(cmd, shell=True).decode("utf-8").strip()
            gpu_util, mem_used, mem_total = map(int, result.split(","))

            gpu_usage.append({
                "gpu_id": i,
                "gpu_util": gpu_util,
                "mem_used": mem_used,
                "mem_total": mem_total,
                "mem_free": mem_total - mem_used,
            })
        except Exception:
            torch.cuda.set_device(i)
            allocated = torch.cuda.memory_allocated(i) / 1024**2

            gpu_usage.append({
                "gpu_id": i,
                "gpu_util": 0,
                "mem_used": allocated,
                "mem_total": 48000,
                "mem_free": 48000 - allocated,
            })

    if gpu_usage:
        gpu_usage.sort(key=lambda x: x["mem_free"], reverse=True)
        return gpu_usage[0]["gpu_id"]

    return 0


OUTPUT_CSV_PATH = "deterministic_full_grid_search.csv"

PARAM_GRID = {
    "enhancer_heads": [1],
    "recon_weight": [0.7],
    "contrastive_weight": [0.5],
}

DATASET_IDS = list(range(151507, 151511)) + list(range(151669, 151677))

DATASET_CLUSTERS = {
    151507: 7,
    151508: 7,
    151509: 7,
    151510: 7,
    151669: 5,
    151670: 5,
    151671: 5,
    151672: 5,
    151673: 7,
    151674: 7,
    151675: 7,
    151676: 7,
}


class TwelveSlicesBestEpochEvaluator:
    def __init__(self):
        self.all_slice_results = {}
        self.all_model_states = {}
        self.best_epoch_overall = -1
        self.best_avg_ari_overall = -1.0
        self.epoch_avg_aris = {}

    def add_slice_result(self, slice_id, epoch, ari, model_state=None):
        if slice_id not in self.all_slice_results:
            self.all_slice_results[slice_id] = {}
            self.all_model_states[slice_id] = {}

        self.all_slice_results[slice_id][epoch] = ari

        if model_state is not None:
            self.all_model_states[slice_id][epoch] = model_state

    def find_best_epoch_for_all_slices(self):
        all_epochs = set()

        for slice_id in self.all_slice_results:
            all_epochs.update(self.all_slice_results[slice_id].keys())

        self.epoch_avg_aris = {}

        for epoch in sorted(all_epochs):
            aris_at_epoch = []
            all_slices_have_epoch = True

            for slice_id in DATASET_IDS:
                if slice_id not in self.all_slice_results:
                    all_slices_have_epoch = False
                    break

                if epoch not in self.all_slice_results[slice_id]:
                    all_slices_have_epoch = False
                    break

                aris_at_epoch.append(self.all_slice_results[slice_id][epoch])

            if all_slices_have_epoch and len(aris_at_epoch) == len(DATASET_IDS):
                self.epoch_avg_aris[epoch] = np.mean(aris_at_epoch)

        if self.epoch_avg_aris:
            self.best_epoch_overall = max(
                self.epoch_avg_aris,
                key=self.epoch_avg_aris.get,
            )
            self.best_avg_ari_overall = self.epoch_avg_aris[self.best_epoch_overall]
            return self.best_epoch_overall, self.best_avg_ari_overall

        return -1, 0.0

    def get_model_state_for_slice(self, slice_id, epoch):
        if (
            slice_id in self.all_model_states
            and epoch in self.all_model_states[slice_id]
        ):
            return self.all_model_states[slice_id][epoch]

        return None


def train_with_parameters_full(params, param_id, total_combinations):
    enhancer_heads = params["enhancer_heads"]
    recon_weight = params["recon_weight"]
    contrastive_weight = params["contrastive_weight"]

    print(f"\n{'=' * 80}")
    print(f"参数组合 {param_id}/{total_combinations}")
    print(
        f"enhancer_heads={enhancer_heads}, "
        f"recon_weight={recon_weight}, "
        f"contrastive_weight={contrastive_weight}"
    )
    print(f"{'=' * 80}")

    evaluator = TwelveSlicesBestEpochEvaluator()
    results = []
    start_time = datetime.now()

    for idx, current_id in enumerate(DATASET_IDS):
        print(
            f"\n[参数组合 {param_id}] "
            f"训练数据集 {current_id} ({idx + 1}/{len(DATASET_IDS)})"
        )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        try:
            seed_everything(SEED)

            file_fold = f"/private_data/summer/liuxy/mydata/1.DLPFC/{current_id}"

            adata = sc.read_visium(
                file_fold,
                count_file="filtered_feature_bc_matrix.h5",
                load_images=True,
            )
            adata.var_names_make_unique()

            metadata = pd.read_csv(f"{file_fold}/metadata.tsv", sep="\t", index_col=0)
            adata.obs["ground_truth"] = metadata["layer_guess"]

            selected_gpu = select_most_idle_gpu()

            if selected_gpu == "cpu":
                device = torch.device("cpu")
                print("使用 CPU 进行训练")
            else:
                device = torch.device(f"cuda:{selected_gpu}")
                torch.cuda.set_device(selected_gpu)
                torch.cuda.empty_cache()
                print(f"使用 GPU{selected_gpu} 进行训练")

            def record_callback(epoch, ari, model_state):
                evaluator.add_slice_result(current_id, epoch, ari, model_state)

                if epoch % 100 == 0:
                    print(f"  切片 {current_id} - Epoch {epoch}: ARI = {ari:.4f}")

            n_clusters = DATASET_CLUSTERS.get(current_id, 7)

            from GraphST.preprocess import preprocess

            preprocess(
                adata,
                use_spatial_smooth=False,
                n_top_genes=3000,
                use_gene_module_filter=True,
                min_module_size=80,
                min_keep_genes=500,
                leiden_resolution=1.5,
                gene_neighbors=10,
                n_pcs=30,
            )

            module_df = adata.uns["gene_module_df"]

            total_modules = module_df["module"].nunique()
            selected_modules = module_df[module_df["selected"]]["module"].nunique()
            selected_genes = int(module_df["selected"].sum())

            selected_module_count = (
                module_df[module_df["selected"]]
                .groupby("module")
                .size()
                .reset_index(name="gene_count")
                .sort_values("gene_count", ascending=False)
            )

            selected_module_count.insert(0, "slice_id", current_id)
            selected_module_count.insert(1, "total_modules", total_modules)
            selected_module_count.insert(2, "selected_modules", selected_modules)
            selected_module_count.insert(3, "selected_genes_for_encoder", selected_genes)

            selected_module_count.to_csv(
                f"selected_gene_module_summary_{current_id}.csv",
                index=False,
            )

            print("总 module 数:", total_modules)
            print("被保留 module 数:", selected_modules)
            print("进入 Encoder 的 gene 数:", selected_genes)
            print(selected_module_count)

            adata_copy = adata.copy()

            print("Final HVG used:", adata_copy.var["highly_variable"].sum())

            adata_copy.obsm.pop("feat", None)
            adata_copy.obsm.pop("feat_a", None)

            model = GraphST.GraphST(
                adata_copy,
                device=device,
                n_clusters=n_clusters,
                dataset_path=file_fold,
                enhancer_heads=enhancer_heads,
                recon_weight=recon_weight,
                contrastive_weight=contrastive_weight,
                epochs=600,
                random_seed=SEED,
            )

            adata_result = model.train_with_callback(record_callback)

            print("emb exist:", "emb" in adata_result.obsm)
            print(f"切片 {current_id} 训练完成")

        except Exception as e:
            print(f"切片 {current_id} 训练失败: {str(e)}")
            import traceback

            traceback.print_exc()
            continue

    best_epoch_overall, best_avg_ari_overall = evaluator.find_best_epoch_for_all_slices()

    if best_epoch_overall == -1:
        print("\n警告：没有找到有效的共同最佳 epoch")
        best_epoch_overall = 300
        best_avg_ari_overall = 0.0

    print(f"\n[参数组合 {param_id}] 12 个切片共同的最佳 epoch: {best_epoch_overall}")
    print(f"[参数组合 {param_id}] 最佳 epoch 下的平均 ARI: {best_avg_ari_overall:.4f}")

    print("各 epoch 平均 ARI:")
    for epoch, avg_ari in sorted(evaluator.epoch_avg_aris.items()):
        print(f"  Epoch {epoch}: {avg_ari:.4f}")

    slice_aris_at_best_epoch = {}

    for current_id in DATASET_IDS:
        if (
            current_id in evaluator.all_slice_results
            and best_epoch_overall in evaluator.all_slice_results[current_id]
        ):
            slice_ari = evaluator.all_slice_results[current_id][best_epoch_overall]
        else:
            slice_ari = 0.0

        slice_aris_at_best_epoch[current_id] = slice_ari
        print(f"切片 {current_id} 在 epoch {best_epoch_overall} 的 ARI: {slice_ari:.4f}")

    for current_id in DATASET_IDS:
        slice_ari = slice_aris_at_best_epoch.get(current_id, 0.0)

        results.append({
            "param_id": param_id,
            "enhancer_heads": enhancer_heads,
            "recon_weight": recon_weight,
            "contrastive_weight": contrastive_weight,
            "slice_id": current_id,
            "best_epoch_overall": best_epoch_overall,
            "slice_ari_at_best_epoch": slice_ari,
            "avg_ari_all_slices": best_avg_ari_overall,
            "status": "Success",
            "training_time": (datetime.now() - start_time).total_seconds(),
            "test_date": datetime.now().strftime("%Y-%m-%d"),
        })

    print(f"\n[参数组合 {param_id}] 完成!")
    print(f"  12 个切片平均 ARI: {best_avg_ari_overall:.4f}")
    print(f"  最佳 epoch: {best_epoch_overall}")
    print(f"  总耗时: {datetime.now() - start_time}")

    return results


def main():
    print("开始严格可复现全参数搜索")
    print(
        f"参数组合数: "
        f"{len(PARAM_GRID['enhancer_heads'])} x "
        f"{len(PARAM_GRID['recon_weight'])} x "
        f"{len(PARAM_GRID['contrastive_weight'])}"
    )
    print(f"数据集: {len(DATASET_IDS)} 个切片")
    print(f"输出文件: {OUTPUT_CSV_PATH}")
    print("=" * 80)

    param_names = list(PARAM_GRID.keys())
    param_values = list(PARAM_GRID.values())
    param_combinations = list(itertools.product(*param_values))

    print(f"总共有 {len(param_combinations)} 种参数组合")
    print("评估 epoch: 200, 250, 300, 350, 400, 450, 500, 550, 600")
    print(f"每个参数组合将训练所有 {len(DATASET_IDS)} 个切片")

    columns = [
        "param_id",
        "enhancer_heads",
        "recon_weight",
        "contrastive_weight",
        "slice_id",
        "best_epoch_overall",
        "slice_ari_at_best_epoch",
        "avg_ari_all_slices",
        "status",
        "training_time",
        "test_date",
    ]

    if not os.path.exists(OUTPUT_CSV_PATH):
        pd.DataFrame(columns=columns).to_csv(OUTPUT_CSV_PATH, index=False)
        print(f"创建新的 CSV 文件: {OUTPUT_CSV_PATH}")
    else:
        print(f"使用现有 CSV 文件: {OUTPUT_CSV_PATH}")

    total_start_time = datetime.now()
    completed_combinations = 0

    completed_params = set()

    if os.path.exists(OUTPUT_CSV_PATH) and os.path.getsize(OUTPUT_CSV_PATH) > 0:
        try:
            existing_results = pd.read_csv(OUTPUT_CSV_PATH)

            if not existing_results.empty:
                for _, row in existing_results.iterrows():
                    param_key = (
                        row["enhancer_heads"],
                        row["recon_weight"],
                        row["contrastive_weight"],
                    )
                    completed_params.add(param_key)

                print(f"已完成的参数组合数: {len(completed_params)}")
        except Exception as e:
            print(f"读取现有结果失败: {str(e)}")

    for idx, param_values in enumerate(param_combinations):
        param_id = idx + 1
        param_key = (param_values[0], param_values[1], param_values[2])

        if param_key in completed_params:
            print(f"\n参数组合 {param_id} 已处理过，跳过")
            continue

        params = {
            "enhancer_heads": param_values[0],
            "recon_weight": param_values[1],
            "contrastive_weight": param_values[2],
        }

        print(f"\n{'=' * 80}")
        print(f"开始处理参数组合 {param_id}/{len(param_combinations)}")
        print(f"参数: {params}")
        print(f"{'=' * 80}")

        try:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            results = train_with_parameters_full(
                params,
                param_id,
                len(param_combinations),
            )

            if results:
                df_batch = pd.DataFrame(results)
                file_exists = (
                    os.path.exists(OUTPUT_CSV_PATH)
                    and os.path.getsize(OUTPUT_CSV_PATH) > 0
                )

                df_batch.to_csv(
                    OUTPUT_CSV_PATH,
                    mode="a",
                    header=not file_exists,
                    index=False,
                    sep=",",
                    quoting=csv.QUOTE_MINIMAL,
                )

                print(f"参数组合 {param_id} 结果已保存到 {OUTPUT_CSV_PATH}")

                completed_combinations += 1

                elapsed_time = (datetime.now() - total_start_time).total_seconds()

                if completed_combinations > 0:
                    avg_time_per_combo = elapsed_time / completed_combinations
                    remaining_combos = len(param_combinations) - param_id
                    estimated_remaining = avg_time_per_combo * remaining_combos

                    print("\n进度统计:")
                    print(f"  已完成: {completed_combinations}/{len(param_combinations)}")
                    print(f"  进度: {param_id / len(param_combinations) * 100:.1f}%")
                    print(f"  已用时间: {elapsed_time / 3600:.2f} 小时")
                    print(f"  预计剩余时间: {estimated_remaining / 3600:.2f} 小时")
                    print(
                        f"  预计总时间: "
                        f"{(elapsed_time + estimated_remaining) / 3600:.2f} 小时"
                    )
            else:
                print(f"参数组合 {param_id} 没有生成结果")

        except Exception as e:
            print(f"参数组合 {param_id} 处理失败: {str(e)}")
            import traceback

            traceback.print_exc()

            error_results = []

            for current_id in DATASET_IDS:
                error_results.append({
                    "param_id": param_id,
                    "enhancer_heads": param_values[0],
                    "recon_weight": param_values[1],
                    "contrastive_weight": param_values[2],
                    "slice_id": current_id,
                    "best_epoch_overall": -1,
                    "slice_ari_at_best_epoch": 0.0,
                    "avg_ari_all_slices": 0.0,
                    "status": f"Error: {str(e)[:200]}",
                    "training_time": 0,
                    "test_date": datetime.now().strftime("%Y-%m-%d"),
                })

            if error_results:
                df_error = pd.DataFrame(error_results)
                file_exists = (
                    os.path.exists(OUTPUT_CSV_PATH)
                    and os.path.getsize(OUTPUT_CSV_PATH) > 0
                )

                df_error.to_csv(
                    OUTPUT_CSV_PATH,
                    mode="a",
                    header=not file_exists,
                    index=False,
                    sep=",",
                    quoting=csv.QUOTE_MINIMAL,
                )

    print(f"\n{'=' * 80}")
    print("严格可复现全参数搜索完成，分析结果")
    print(f"{'=' * 80}")

    if os.path.exists(OUTPUT_CSV_PATH):
        all_results = pd.read_csv(OUTPUT_CSV_PATH)

        if not all_results.empty:
            param_summary = (
                all_results
                .groupby([
                    "param_id",
                    "enhancer_heads",
                    "recon_weight",
                    "contrastive_weight",
                ])
                .agg({
                    "avg_ari_all_slices": "first",
                    "slice_ari_at_best_epoch": "mean",
                    "status": lambda x: (
                        "Success" if all(s == "Success" for s in x) else "Partial"
                    ),
                })
                .reset_index()
            )

            successful_params = param_summary[param_summary["status"] == "Success"]

            if not successful_params.empty:
                best_param = successful_params.loc[
                    successful_params["avg_ari_all_slices"].idxmax()
                ]

                print("\n最佳参数组合:")
                print(f"  enhancer_heads: {best_param['enhancer_heads']}")
                print(f"  recon_weight: {best_param['recon_weight']}")
                print(f"  contrastive_weight: {best_param['contrastive_weight']}")
                print(f"  12 个切片平均 ARI: {best_param['avg_ari_all_slices']:.4f}")
                print(f"  切片平均 ARI: {best_param['slice_ari_at_best_epoch']:.4f}")

                summary_df = pd.DataFrame([{
                    "best_enhancer_heads": best_param["enhancer_heads"],
                    "best_recon_weight": best_param["recon_weight"],
                    "best_contrastive_weight": best_param["contrastive_weight"],
                    "best_12slices_avg_ari": best_param["avg_ari_all_slices"],
                    "best_slice_avg_ari": best_param["slice_ari_at_best_epoch"],
                    "total_param_combinations": len(param_combinations),
                    "successful_combinations": len(successful_params),
                    "total_slices": len(DATASET_IDS),
                    "total_training_time_hours": (
                        datetime.now() - total_start_time
                    ).total_seconds() / 3600,
                    "analysis_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                }])

                summary_file = "best_parameters_full_search_summary.csv"
                summary_df.to_csv(summary_file, index=False)
                print(f"最佳参数总结已保存到 {summary_file}")

                print("\n前 10 个最佳参数组合:")
                print("-" * 90)
                print(
                    f"{'排名':<4} {'heads':<6} {'recon':<6} "
                    f"{'contrast':<8} {'12切片ARI':<10} {'切片平均':<10}"
                )
                print("-" * 90)

                top_params = (
                    successful_params
                    .sort_values("avg_ari_all_slices", ascending=False)
                    .head(10)
                )

                for i, row in top_params.iterrows():
                    print(
                        f"{i + 1:<4} "
                        f"{row['enhancer_heads']:<6} "
                        f"{row['recon_weight']:<6.1f} "
                        f"{row['contrastive_weight']:<8.1f} "
                        f"{row['avg_ari_all_slices']:<10.4f} "
                        f"{row['slice_ari_at_best_epoch']:<10.4f}"
                    )

                ranking_file = "parameter_ranking_full_search.csv"
                top_params.to_csv(ranking_file, index=False)
                print(f"详细排名已保存到 {ranking_file}")

                print("\n统计信息:")
                print(f"  总参数组合数: {len(param_combinations)}")
                print(f"  成功组合数: {len(successful_params)}")
                print(f"  失败组合数: {len(param_summary) - len(successful_params)}")
                print(
                    f"  最佳 12 切片 ARI 范围: "
                    f"[{successful_params['avg_ari_all_slices'].min():.4f}, "
                    f"{successful_params['avg_ari_all_slices'].max():.4f}]"
                )
                print(
                    f"  平均 12 切片 ARI: "
                    f"{successful_params['avg_ari_all_slices'].mean():.4f}"
                )
                print(f"  总耗时: {datetime.now() - total_start_time}")
            else:
                print("没有成功的参数组合")
        else:
            print("结果文件为空")
    else:
        print(f"没有找到结果文件: {OUTPUT_CSV_PATH}")

    print("\n严格可复现全参数搜索完成!")
    print(f"  开始时间: {total_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  结束时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"  总耗时: {datetime.now() - total_start_time}")


if __name__ == "__main__":
    main()

/private_data/summer/liuxy/miniconda3/envs/graphst/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


开始严格可复现全参数搜索
参数组合数: 1 x 1 x 1
数据集: 12 个切片
输出文件: deterministic_full_grid_search.csv
总共有 1 种参数组合
评估 epoch: 200, 250, 300, 350, 400, 450, 500, 550, 600
每个参数组合将训练所有 12 个切片
使用现有 CSV 文件: deterministic_full_grid_search.csv

开始处理参数组合 1/1
参数: {'enhancer_heads': 1, 'recon_weight': 0.7, 'contrastive_weight': 0.5}

参数组合 1/1
enhancer_heads=1, recon_weight=0.7, contrastive_weight=0.5

[参数组合 1] 训练数据集 151507 (1/12)
使用 GPU0 进行训练
[Preprocess] skip spatial smoothing.
[GeneModule-Leiden] modules=28, large_modules=19, selected_genes=2530
[Preprocess] final selected genes for Encoder: 2530
总 module 数: 28
被保留 module 数: 19
进入 Encoder 的 gene 数: 2530
    slice_id  total_modules  selected_modules  selected_genes_for_encoder  \
0     151507             28                19                        2530   
1     151507             28                19                        2530   
2     151507             28                19                        2530   
3     151507             28                19              

 33%|███▎      | 199/600 [03:02<05:55,  1.13it/s]R callback write-console:                    __           __ 
   ____ ___  _____/ /_  _______/ /_
  / __ `__ \/ ___/ / / / / ___/ __/
 / / / / / / /__/ / /_/ (__  ) /_  
/_/ /_/ /_/\___/_/\__,_/____/\__/   version 6.1.2
Type 'citation("mclust")' for citing this R package in publications.
  


<class 'pandas.core.frame.DataFrame'>
fitting ...
  |======================================================================| 100%


 33%|███▎      | 200/600 [03:17<33:27,  5.02s/it]

...

 42%|████▏     | 249/600 [04:01<05:16,  1.11it/s]

<class 'pandas.core.frame.DataFrame'>
fitting ...
  |======================================================================| 100%


 42%|████▏     | 250/600 [04:18<33:18,  5.71s/it]

Epoch 250: ARI = 0.5311


 49%|████▉     | 293/600 [04:56<04:42,  1.09it/s]